# Chapter 37: Trees, Random Forests, and Gradient Boosting

Synthetic NRG shipment risk illustrates complexity, ensembling, and validation.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, log_loss
sys.path.insert(0,str(Path.cwd().parents[1]/'src'))
from datasciencebook.tree_ensembles import fit_with_seed, classification_report, leaf_profile, permutation_scores
print('Imports ready.')


Imports ready.


In [ ]:
rng=np.random.default_rng(37); n=600
distance=rng.uniform(100,2500,n); wait=rng.gamma(2,3,n); refrigerated=rng.integers(0,2,n)
logit=-4+0.0012*distance+0.18*wait+0.9*refrigerated*(wait>7)
prob=1/(1+np.exp(-logit)); late=rng.binomial(1,prob); x=np.column_stack([distance,wait,refrigerated]); cut=450
print(f'Train rows: {cut}; test rows: {n-cut}; test prevalence: {late[cut:].mean():.1%}')


Train rows: 450; test rows: 150; test prevalence: 30.0%


In [ ]:
tree=fit_with_seed(DecisionTreeClassifier(max_depth=3,min_samples_leaf=18),x[:cut],late[:cut])
forest=fit_with_seed(RandomForestClassifier(n_estimators=180,max_depth=7,min_samples_leaf=8,max_features='sqrt'),x[:cut],late[:cut])
boost=fit_with_seed(HistGradientBoostingClassifier(max_iter=120,max_leaf_nodes=9,learning_rate=.06,l2_regularization=1),x[:cut],late[:cut])
for name,model in [('tree',tree),('forest',forest),('boost',boost)]:
 p=model.predict_proba(x[cut:])[:,1]; print(name, f'AUC={roc_auc_score(late[cut:],p):.3f}', f'logloss={log_loss(late[cut:],p):.3f}')


tree AUC=0.742 logloss=0.546
forest AUC=0.804 logloss=0.484
boost AUC=0.768 logloss=0.522


In [ ]:
print('Tree profile:',leaf_profile(tree))
importance=permutation_scores(forest,x[cut:],late[cut:],'roc_auc',repeats=8)
importance_report={name:round(float(value),3) for name,value in zip(['distance','wait','refrigerated'],importance)}
print('Forest permutation importance:',importance_report)


Tree profile: {'depth': 3, 'leaves': 7, 'smallest_leaf': 18}
Forest permutation importance: {'distance': 0.154, 'wait': 0.168, 'refrigerated': 0.019}


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
plot_tree(tree,feature_names=['distance','wait','refrigerated'],class_names=['on time','late'],filled=True,fontsize=7,ax=axes[0])
axes[0].set_title('Constrained decision tree')
axes[1].bar(['distance','wait','refrigerated'],importance,color='#2a6f97'); axes[1].set(ylabel='ROC AUC decrease',title='Validation permutation importance')
fig.tight_layout(); plt.show()


## Interpretation

The constrained tree offers a readable baseline. Forest and boosting results must be judged on unseen metrics and decision cost. Permutation importance is predictive reliance under the evaluated data, not a causal effect.


In [ ]:
# Practice: vary leaf size and compare unseen AUC, log loss, and stability.
